# FungiCirno — oyster model training (Google Colab)

Upload/extract this FungiCirno release into `MyDrive/FungiCirno` before running the notebook. It trains the two production model families using the project's existing scripts. It does **not** download datasets automatically: obtain the public Charisis dataset (DOI `10.5281/zenodo.15041243`) and the annotated MushR dataset linked from `https://github.com/ETCE-LAB/MushR`, verify their terms, and upload/mount them in Drive.

- **Model A:** whole oyster cluster stages (`immature`, `well-defined`, `overstayed`) from Charisis time-lapse data. Compare M18K WB initialization against COCO YOLOv8-seg initialization.
- **Model B:** individual oyster fruitbody maturity (`not-ready`, `ready`, `overdue`) from MushR.
- Keep train/validation/test splits independent by complete sequence/trial/camera group. Never randomly split adjacent frames.
- The untouched test split is used only for evaluation. Do not tune against it repeatedly.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/FungiCirno')
if not (PROJECT / 'tools' / 'train_candidate.py').is_file():
    raise FileNotFoundError('Upload/extract this FungiCirno release into MyDrive/FungiCirno first')
%cd /content/drive/MyDrive/FungiCirno


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-research.txt


## 1. Configure source dataset paths

The generic converter requires COCO instance-segmentation JSON files for train, validation, and test, plus a common image root. If a dataset provides a different annotation format, inspect its metadata and use a dataset-specific conversion; do not relabel or randomly repartition it blindly.


In [ ]:
# Replace these with the paths in your Drive dataset folders.
CHARISIS_IMAGES = '/content/drive/MyDrive/datasets/charisis/images'
CHARISIS_TRAIN_JSON = '/content/drive/MyDrive/datasets/charisis/annotations/train.json'
CHARISIS_VAL_JSON = '/content/drive/MyDrive/datasets/charisis/annotations/val.json'
CHARISIS_TEST_JSON = '/content/drive/MyDrive/datasets/charisis/annotations/test.json'
CHARISIS_OUT = '/content/charisis_yolo'

MUSHR_IMAGES = '/content/drive/MyDrive/datasets/mushr/images'
MUSHR_TRAIN_JSON = '/content/drive/MyDrive/datasets/mushr/annotations/train.json'
MUSHR_VAL_JSON = '/content/drive/MyDrive/datasets/mushr/annotations/val.json'
MUSHR_TEST_JSON = '/content/drive/MyDrive/datasets/mushr/annotations/test.json'
MUSHR_OUT = '/content/mushr_yolo'

M18K_BEST = '/content/drive/MyDrive/models/m18k_wb_best.pt'


## 2. Convert Charisis annotations (if COCO-compatible)

Confirm that the JSON category names refer to the Charisis cluster classes. The converter enforces the class order and emits a manifest. It keeps the supplied splits separate; it does not prove that your source split is sequence-disjoint, so inspect its group metadata before continuing.


In [ ]:
!python tools/prepare_coco_yolo_dataset.py \
  --images "$CHARISIS_IMAGES" \
  --train-json "$CHARISIS_TRAIN_JSON" \
  --val-json "$CHARISIS_VAL_JSON" \
  --test-json "$CHARISIS_TEST_JSON" \
  --class-order immature,well-defined,overstayed \
  --out "$CHARISIS_OUT"


## 3. Train Model A — compare initializations

Run these separately on the same train/validation/test files. The M18K class head differs from the target three-stage head; Ultralytics transfers compatible parameters and adapts the task head. Record the training log and compare **target oyster test metrics**, not M18K source metrics.


In [ ]:
!python tools/train_candidate.py \
  --weights "$M18K_BEST" \
  --data "$CHARISIS_OUT/data.yaml" \
  --holdout-data "$CHARISIS_OUT/data.yaml" --holdout-split test \
  --holdout-id charisis_trial_holdout_v1 \
  --model-key charisis_cluster_tracking \
  --species 'Pleurotus ostreatus' \
  --taxonomy charisis_oyster_cluster_stage_v1 \
  --prediction-role cluster_stage \
  --pretrained-source 'team M18K V2 WB-only YOLOv8n-seg' \
  --epochs 50 --imgsz 640 --batch 8 --device 0 \
  --out /content/charisis_m18k_init_candidate.zip


In [ ]:
!python tools/train_candidate.py \
  --weights yolov8n-seg.pt \
  --data "$CHARISIS_OUT/data.yaml" \
  --holdout-data "$CHARISIS_OUT/data.yaml" --holdout-split test \
  --holdout-id charisis_trial_holdout_v1 \
  --model-key charisis_cluster_tracking \
  --species 'Pleurotus ostreatus' \
  --taxonomy charisis_oyster_cluster_stage_v1 \
  --prediction-role cluster_stage \
  --pretrained-source 'Ultralytics COCO-pretrained YOLOv8n-seg' \
  --epochs 50 --imgsz 640 --batch 8 --device 0 \
  --out /content/charisis_coco_init_candidate.zip


**Selection rule:** compare the two candidate `evaluation_metrics.json` files and qualitative overlays on the same untouched test sequences. Pick the stronger target-domain initialization after checking per-class metrics and errors. First-transfer runs intentionally omit `--baseline-weights`: WB and oyster-stage taxonomies are not comparable.


## 4. Convert MushR and train Model B

MushR's official repository links its annotated oyster dataset. Inspect the downloaded archive structure first. If the annotation file is COCO-compatible, run the conversion below; otherwise build/validate a format-specific converter before training.


In [ ]:
!python tools/prepare_coco_yolo_dataset.py \
  --images "$MUSHR_IMAGES" \
  --train-json "$MUSHR_TRAIN_JSON" \
  --val-json "$MUSHR_VAL_JSON" \
  --test-json "$MUSHR_TEST_JSON" \
  --class-order not-ready,ready,overdue \
  --out "$MUSHR_OUT"


In [ ]:
!python tools/train_candidate.py \
  --weights yolov8n-seg.pt \
  --data "$MUSHR_OUT/data.yaml" \
  --holdout-data "$MUSHR_OUT/data.yaml" --holdout-split test \
  --holdout-id mushr_test_v1 \
  --model-key mushr_maturity \
  --species 'Oyster mushroom (MushR dataset)' \
  --taxonomy mushr_oyster_fruitbody_maturity_v1 \
  --prediction-role fruitbody_maturity \
  --pretrained-source 'Ultralytics COCO-pretrained YOLOv8n-seg' \
  --epochs 50 --imgsz 640 --batch 8 --device 0 \
  --out /content/mushr_candidate.zip


## 5. Preserve artifacts and report actual results

Copy candidate ZIPs, Ultralytics training runs, split manifests, evaluation JSON, plots and sample predictions into the project results folder in Drive. Keep raw datasets and checkpoints outside Git. Do not call either model production-ready until the local application has been run on CPU and, separately, AMD DirectML with the real bundle.
